In [1]:
import os
import pandas as pd
from pathlib import Path

In [ ]:
RAW_TRAINING_DATA_PATH = Path(os.getcwd()).parent / "data/raw/fraudTrain.csv"
RAW_TESTING_DATA_PATH = Path(os.getcwd()).parent / "data/raw/fraudTest.csv"
SAVE_TRAINING_DATA_PATH = Path(os.getcwd()).parent / "data/processed/fraudTrain.csv"
SAVE_TESTING_DATA_PATH = Path(os.getcwd()).parent / "data/processed/fraudTest.csv"

In [3]:
# Load the training and testing data

train_total_df = pd.read_csv(RAW_TRAINING_DATA_PATH)
test_total_df = pd.read_csv(RAW_TESTING_DATA_PATH)

print(f"Training samples: {len(train_total_df)}, Testing samples: {len(test_total_df)}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/ivanextrastuff/code/ml/machine_learning/ai/data/raw/fraud_train.csv'

In [ ]:
# Randomly shuffle and sample data

TRAIN_SIZE = 10000
TEST_SIZE = 1000
RANDOM_SEED = 42

fraudulent_transactions_train = train_total_df[train_total_df['is_fraud'] == 1].sample(TRAIN_SIZE // 2, random_state=RANDOM_SEED)
legitimate_transactions_train = train_total_df[train_total_df['is_fraud'] == 0].sample(TRAIN_SIZE // 2, random_state=RANDOM_SEED)
fraudulent_transactions_test = test_total_df[test_total_df['is_fraud'] == 1].sample(TEST_SIZE // 2, random_state=RANDOM_SEED)
legitimate_transactions_test = test_total_df[test_total_df['is_fraud'] == 0].sample(TEST_SIZE // 2, random_state=RANDOM_SEED)

train_df = pd.concat([fraudulent_transactions_train, legitimate_transactions_train], axis=0).reset_index(drop=True)
test_df = pd.concat([fraudulent_transactions_test, legitimate_transactions_test], axis=0).reset_index(drop=True)

In [ ]:
# Extract features and labels

FEATURES = ['category', 'amt', 'unix_time', 'is_fraud']

# Select only the relevant features
train_df = train_df[FEATURES]
test_df = test_df[FEATURES]

# Get hour of the transaction from the unix time
train_df['hour'] = pd.to_datetime(train_df['unix_time'], unit='s').dt.hour
test_df['hour'] = pd.to_datetime(test_df['unix_time'], unit='s').dt.hour
train_df.drop(columns=['unix_time'], inplace=True)
test_df.drop(columns=['unix_time'], inplace=True)

# One-hot encode the transaction categories
train_df = pd.get_dummies(train_df, columns=['category']).astype(int)
test_df = pd.get_dummies(test_df, columns=['category']).astype(int)

test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

In [ ]:
# Save the processed data

train_df.to_csv(SAVE_TRAINING_DATA_PATH, index=False)
test_df.to_csv(SAVE_TESTING_DATA_PATH, index=False)